# Train YOLOv9-s slim for the Coral EdgeTPU (ReLU6, 17 COCO classes)

This notebook trains the model family behind this release's Coral model: YOLOv9-s with **ReLU6** activations and a
**slim detection head** (`DualDDetectSlim`, arch `distlite_cls64`). Convert the result with
`Convert_YOLOv9s_slim_to_EdgeTPU.ipynb`.

**The head.** YOLOv9's `DualDDetect` has two branches. Only the main branch is exported to the EdgeTPU; the aux branch
is deep supervision during training and is dropped at export. The slim head changes only the main branch:

| main-branch tower | stock DualDDetect | slim `distlite_cls64` |
|---|---|---|
| box (`cv4`) | 3×3(64) → 3×3(64, g4) → 1×1 → 64 DFL bins | 3×3(32) → 1×1 → **4 ltrb distances** (stride units) |
| class (`cv5`) | 3×3(128) → 3×3(128) → 1×1 → 17 | 3×3(64) → 3×3(64) → 1×1 → 17 |

The box loss for the distlite branch is CIoU on `clamp(d, 0)`-decoded boxes plus smooth-L1 on the raw distances (in
the DFL slot). On a Coral this head is about 30 % faster end to end (6.25 ms vs about 9 ms per frame) at about the
same accuracy; see `benchmark/RESULTS.md`.

**Recipe (two stages):**
1. **Init**: a fresh slim model gets every matching weight (backbone, neck, aux head) from a trained 17-class ReLU6
   checkpoint, by default this repo's v1.0 `yolov9-s-relu6-best.pt`. The slim towers start random.
2. **Stage A** (2 epochs, backbone layers 0–8 frozen, `lr0` 0.005): trains the new towers without disturbing the backbone.
3. **Stage B** (one run of `EPOCHS_TOTAL` epochs, all layers, `lr0` 0.002 → `lrf` 0.05, linear LR, no mosaic).

### Rules that keep a long run correct
Colab and Kaggle sessions end after a few hours, so stage B normally takes several sessions ("bouts"). The schedule
only stays correct if every bout continues the **same** run:
- **Continue with a true resume** (`--resume .../weights/last.pt`). It restores the optimizer, EMA, epoch and LR
  schedule. **Never** start a new run with `--weights last.pt`: that resets the LR warm-up and momentum, and accuracy
  dips every time.
- **Freeze `EPOCHS_TOTAL`.** The LR curve is computed from the total. Changing it in the middle of a run stretches the
  curve. The resume cell refuses a run whose recorded total differs.
- **Keep the run folder on persistent storage** (Google Drive below) at the same path, because `opt.yaml` stores it.
  An interrupted session leaves a resumable `last.pt` from the last finished epoch; re-run the stage B cell to continue.
- When a run finishes, YOLOv9 strips the optimizer out of `last.pt`. The wrapper saves a full copy first as
  **`last_resume.pt`**. Package that file (plus `best.pt`); a stripped `last.pt` cannot be resumed.

### What this notebook does not include
The released model was trained on a mixed set of COCO and other day, night and IR sources, with an augmentation
that replaces half of the training images with Frigate-style motion crops, over several bouts. Neither that data set
nor the crop augmentation is part of this notebook. By default it trains on COCO 2017 filtered to the 17 classes, so
expect different numbers. A hard-swish variant needs EdgeTPU NLU patching and is not covered either.


## Step 1: GPU and storage


In [ ]:
import os, sys, glob, json, subprocess, shutil
from pathlib import Path
r = subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True)
print(r.stdout or '⚠️  no GPU: Runtime > Change runtime type > GPU')
USE_DRIVE = True   # keep runs on Google Drive so a lost session can resume
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive')
    except ImportError:
        print('not on Colab: set RUN_ROOT below to a persistent folder'); USE_DRIVE = False


## Step 2: Options


In [ ]:
RUN_ROOT = '/content/drive/MyDrive/coral_slim_runs' if USE_DRIVE else '/content/runs'   # persistent!
RUN_NAME = 'yolov9s_relu6_distlite'   # stage A = <RUN_NAME>_A, stage B = <RUN_NAME>_B
SLIM = 'distlite_cls64'               # released head; 'cls64' = slim class tower + stock DFL box tower
EPOCHS_TOTAL = 40                     # stage B length. Freeze it once stage B has started.
STAGE_A_EPOCHS = 2                    # head warm-up with backbone layers 0-8 frozen
IMG = 320
BATCH = 64                            # T4 16 GB at 320; use 32 if CUDA runs out of memory
WORKERS = 4
DEVICE = '0'                          # GPU id, or 'cpu' for a smoke test
INIT_SRC = 'https://github.com/dbro/frigate-detector-edgetpu-yolo9/releases/download/v1.0/yolov9-s-relu6-best.pt'
DATA_YAML = ''                        # your own YOLO-format data.yaml with the 17 classes in LABELS order; '' = build COCO-17
COCO_DIR = '/content/coco'            # COCO 2017 is downloaded and filtered here (~20 GB); the folder must be named coco
LABELS = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat',
          'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear']
COCO80_IDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 14, 15, 16, 17, 18, 19, 20, 21]   # COCO-80 index of each label above

WORK = '/content'; REPO = f'{WORK}/yolov9'; RECIPE = f'{WORK}/recipe'; PY = sys.executable
YOLOV9_COMMIT = '5b1ea9a'
os.makedirs(RUN_ROOT, exist_ok=True); os.makedirs(RECIPE, exist_ok=True)


## Step 3: YOLOv9 code and requirements


In [ ]:
if not os.path.isdir(f'{REPO}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/WongKinYiu/yolov9.git', REPO], check=True)
subprocess.run(['git', '-C', REPO, 'checkout', '-q', YOLOV9_COMMIT], check=True)


In [ ]:
# [colab-only]
!pip install -q -r /content/yolov9/requirements.txt


## Step 4: Recipe files

The yolov9 repo stays unmodified; these files patch it at run time.
- `coral_slim.py`: the slim head (`DualDDetectSlim`) and its loss. The convert notebook needs the same file to load checkpoints.
- `coral_patch.py`: ReLU6 everywhere, `torch.load` and Pillow fixes, installs the slim head.
- `coral_init.py`: builds the starting checkpoint.
- `coral_train.py`: wraps `train_dual.py` (safe resume, keeps `last_resume.pt`).
- `coral_val.py`: wraps `val_dual.py`.


In [ ]:
%%writefile /content/recipe/coral_slim.py
"""Train-time slim DualDDetect heads for YOLOv9-s on the Coral EdgeTPU, plus the matching loss.
The yolov9 repo stays unmodified.

install(arch) rebinds models.yolo.DualDDetect -> DualDDetectSlim (a subclass, so parse_model's lookup, the isinstance
checks in Model.__init__ and the yaml name all resolve to it) and utils.loss_tal_dual.ComputeLoss -> ComputeLossSlim.
It must run BEFORE the model is built and BEFORE train_dual.py runs (train_dual does
`from utils.loss_tal_dual import ComputeLoss` at import time). Checkpoints pickle the head as
coral_slim.DualDDetectSlim, so this file must be importable to load them later (val, export).

Only the MAIN branch (cv4 box / cv5 cls, the one exported to the EdgeTPU) is slimmed. The aux branch (cv2/cv3,
training-only deep supervision, dropped at export) keeps the stock towers and the stock DFL loss.
  cls64          : cv5 = Conv(c,64,3) Conv(64,64,3) 1x1->nc;  cv4 = stock (Conv 64, Conv 64 g4, 1x1 g4 -> 4x16 DFL)
  distlite_cls64 : cv5 as cls64;  cv4 = Conv(c,32,3) 1x1 -> 4 (direct ltrb distances in grid/stride units)
dist loss (main branch, replaces DFL): CIoU on boxes decoded from clamp(d, min=0) (IoU of an inverted box is undefined)
+ smooth-L1 (beta 1, mean over 4 sides) of RAW d vs the GT ltrb distance (TAL positives have the anchor inside the GT,
so targets >= 0), weighted by the TAL target score like DFL, in the DFL slot (gain 1.5).
The eval forward decodes dist with clamp(min=0) too; the exported model's host decode must do the same."""
import torch, torch.nn as nn, torch.nn.functional as F
import models.yolo as Y
import utils.loss_tal_dual as L
from models.common import Conv
from utils.tal.anchor_generator import make_anchors, dist2bbox, bbox2dist
from utils.torch_utils import de_parallel

SLIM = {'cls64':          dict(box=[(64, 3, 1), (64, 3, 4)], cls=[(64, 3, 1), (64, 3, 1)], boxout='dfl'),
        'distlite_cls64': dict(box=[(32, 3, 1)],             cls=[(64, 3, 1), (64, 3, 1)], boxout='dist')}
_BASE = Y.DualDDetect if not getattr(Y.DualDDetect, '_coral_slim', False) else Y.DualDDetect.__mro__[1]
_BASE_LOSS = L.ComputeLoss if not getattr(L.ComputeLoss, '_coral_slim', False) else L.ComputeLoss.__mro__[1]


def _tower(cin, hidden, cout, gfinal):
    m, c = [], cin
    for o, k, g in hidden: m.append(Conv(c, o, k, g=g)); c = o
    m.append(nn.Conv2d(c, cout, 1, groups=gfinal)); return nn.Sequential(*m)


class DualDDetectSlim(_BASE):
    _coral_slim = True
    ARCH = None   # set by install(); copied to the instance (pickled with it)

    def __init__(self, nc=80, ch=(), inplace=True):
        super().__init__(nc, ch, inplace)
        arch = self.slim_arch = type(self).ARCH; A = SLIM[arch]; self.boxout = A['boxout']
        self.nbox2 = 4 * self.reg_max if self.boxout == 'dfl' else 4
        self.no2 = nc + self.nbox2
        self.cv4 = nn.ModuleList(_tower(x, A['box'], self.nbox2, 4 if self.boxout == 'dfl' else 1) for x in ch[self.nl:])
        self.cv5 = nn.ModuleList(_tower(x, A['cls'], self.nc, 1) for x in ch[self.nl:])

    def forward(self, x):
        shape = x[0].shape
        d1, d2 = [], []
        for i in range(self.nl):
            d1.append(torch.cat((self.cv2[i](x[i]), self.cv3[i](x[i])), 1))
            d2.append(torch.cat((self.cv4[i](x[self.nl + i]), self.cv5[i](x[self.nl + i])), 1))
        if self.training:
            return [d1, d2]
        elif self.dynamic or self.shape != shape:
            self.anchors, self.strides = (t.transpose(0, 1) for t in make_anchors(d1, self.stride, 0.5))
            self.shape = shape
        box, cls = torch.cat([di.view(shape[0], self.no, -1) for di in d1], 2).split((self.reg_max * 4, self.nc), 1)
        dbox = dist2bbox(self.dfl(box), self.anchors.unsqueeze(0), xywh=True, dim=1) * self.strides
        box2, cls2 = torch.cat([di.view(shape[0], self.no2, -1) for di in d2], 2).split((self.nbox2, self.nc), 1)
        dist2 = self.dfl2(box2) if self.boxout == 'dfl' else box2.clamp(min=0)
        dbox2 = dist2bbox(dist2, self.anchors.unsqueeze(0), xywh=True, dim=1) * self.strides
        y = [torch.cat((dbox, cls.sigmoid()), 1), torch.cat((dbox2, cls2.sigmoid()), 1)]
        return y if self.export else (y, [d1, d2])


class DistBboxLoss(nn.Module):
    """BboxLoss for direct ltrb distances: CIoU + smooth-L1 on distances (in the DFL slot)."""
    def forward(self, pred_dist, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum, fg_mask):
        bbox_mask = fg_mask.unsqueeze(-1).repeat([1, 1, 4])
        pb = torch.masked_select(pred_bboxes, bbox_mask).view(-1, 4)
        tb = torch.masked_select(target_bboxes, bbox_mask).view(-1, 4)
        w = torch.masked_select(target_scores.sum(-1), fg_mask).unsqueeze(-1)
        iou = L.bbox_iou(pb, tb, xywh=False, CIoU=True)
        loss_iou = ((1.0 - iou) * w).sum() / target_scores_sum
        x1y1, x2y2 = torch.split(target_bboxes, 2, -1)
        t_ltrb = torch.cat((anchor_points - x1y1, x2y2 - anchor_points), -1)
        pd = torch.masked_select(pred_dist, bbox_mask).view(-1, 4).float()
        td = torch.masked_select(t_ltrb, bbox_mask).view(-1, 4).float()
        loss_l1 = (F.smooth_l1_loss(pd, td, beta=1.0, reduction='none').mean(-1, keepdim=True) * w).sum() / target_scores_sum
        return loss_iou, loss_l1, iou


class ComputeLossSlim(_BASE_LOSS):
    """ComputeLoss (loss_tal_dual) with a per-branch channel split; main branch 'dist' -> DistBboxLoss."""
    _coral_slim = True

    def __init__(self, model, use_dfl=True):
        super().__init__(model, use_dfl)
        m = de_parallel(model).model[-1]
        self.no2, self.nbox2, self.dist2 = m.no2, m.nbox2, m.boxout == 'dist'
        if self.dist2: self.bbox_loss2 = DistBboxLoss().to(self.device)

    def __call__(self, p, targets, img=None, epoch=0):
        loss = torch.zeros(3, device=self.device)
        feats = p[1][0] if isinstance(p, tuple) else p[0]
        feats2 = p[1][1] if isinstance(p, tuple) else p[1]
        pred_distri, pred_scores = torch.cat([xi.view(feats[0].shape[0], self.no, -1) for xi in feats], 2).split(
            (self.reg_max * 4, self.nc), 1)
        pred_scores = pred_scores.permute(0, 2, 1).contiguous(); pred_distri = pred_distri.permute(0, 2, 1).contiguous()
        pred_distri2, pred_scores2 = torch.cat([xi.view(feats2[0].shape[0], self.no2, -1) for xi in feats2], 2).split(
            (self.nbox2, self.nc), 1)
        pred_scores2 = pred_scores2.permute(0, 2, 1).contiguous(); pred_distri2 = pred_distri2.permute(0, 2, 1).contiguous()
        dtype = pred_scores.dtype
        batch_size = pred_scores.shape[0]
        imgsz = torch.tensor(feats[0].shape[2:], device=self.device, dtype=dtype) * self.stride[0]
        anchor_points, stride_tensor = make_anchors(feats, self.stride, 0.5)
        targets = self.preprocess(targets, batch_size, scale_tensor=imgsz[[1, 0, 1, 0]])
        gt_labels, gt_bboxes = targets.split((1, 4), 2)
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0)
        pred_bboxes = self.bbox_decode(anchor_points, pred_distri)
        pred_bboxes2 = (dist2bbox(pred_distri2.clamp(min=0), anchor_points, xywh=False) if self.dist2
                        else self.bbox_decode(anchor_points, pred_distri2))
        target_labels, target_bboxes, target_scores, fg_mask = self.assigner(
            pred_scores.detach().sigmoid(), (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
            anchor_points * stride_tensor, gt_labels, gt_bboxes, mask_gt)
        target_labels2, target_bboxes2, target_scores2, fg_mask2 = self.assigner2(
            pred_scores2.detach().sigmoid(), (pred_bboxes2.detach() * stride_tensor).type(gt_bboxes.dtype),
            anchor_points * stride_tensor, gt_labels, gt_bboxes, mask_gt)
        target_bboxes /= stride_tensor; target_scores_sum = max(target_scores.sum(), 1)
        target_bboxes2 /= stride_tensor; target_scores_sum2 = max(target_scores2.sum(), 1)
        loss[1] = self.BCEcls(pred_scores, target_scores.to(dtype)).sum() / target_scores_sum
        loss[1] *= 0.25
        loss[1] += self.BCEcls(pred_scores2, target_scores2.to(dtype)).sum() / target_scores_sum2
        if fg_mask.sum():
            loss[0], loss[2], _ = self.bbox_loss(pred_distri, pred_bboxes, anchor_points, target_bboxes, target_scores,
                                                 target_scores_sum, fg_mask)
            loss[0] *= 0.25; loss[2] *= 0.25
        if fg_mask2.sum():
            l0, l2, _ = self.bbox_loss2(pred_distri2, pred_bboxes2, anchor_points, target_bboxes2, target_scores2,
                                        target_scores_sum2, fg_mask2)
            loss[0] += l0; loss[2] += l2
        loss[0] *= 7.5; loss[1] *= 0.5; loss[2] *= 1.5
        return loss.sum() * batch_size, loss.detach()


def install(arch):
    if arch not in SLIM: raise ValueError(f'slim arch must be one of {list(SLIM)}')
    DualDDetectSlim.ARCH = arch
    Y.DualDDetect = DualDDetectSlim
    L.ComputeLoss = ComputeLossSlim
    print(f'[coral_slim] DualDDetect -> DualDDetectSlim({arch}), ComputeLoss -> ComputeLossSlim', flush=True)
    return arch


In [ ]:
%%writefile /content/recipe/coral_patch.py
"""Shared set-up for the coral_* scripts. The yolov9 repo itself is never edited.
setup(repo, slim) must run before any yolov9 model is built or loaded:
  * cwd + sys.path -> repo; warns if the checkout is not the tested commit
  * torch>=2.6: torch.load(weights_only=False) for yolov9 checkpoints (full pickles: trusted files only)
  * Pillow>=10: FreeTypeFont.getsize shim (used by yolov9 utils/plots.py)
  * every yolov9 activation -> ReLU6 (maps fully to the EdgeTPU; SiLU does not)
  * slim: installs coral_slim (DualDDetectSlim head + ComputeLossSlim)"""
import os, subprocess, sys
YOLOV9_COMMIT = '5b1ea9a'   # WongKinYiu/yolov9 commit this recipe was tested with

def setup(repo, slim=''):
    repo = os.path.abspath(repo); os.chdir(repo)
    if repo not in sys.path: sys.path.insert(0, repo)
    head = subprocess.run(['git', '-C', repo, 'rev-parse', '--short=7', 'HEAD'], capture_output=True, text=True).stdout.strip()
    if head != YOLOV9_COMMIT: print(f'WARNING: yolov9 at {head}, recipe tested at {YOLOV9_COMMIT}', flush=True)
    import torch
    if not getattr(torch.load, '_coral', False):
        _load = torch.load
        def load(*a, **k):
            k.setdefault('weights_only', False); return _load(*a, **k)
        load._coral = True; torch.load = load
    from PIL import ImageFont
    if not hasattr(ImageFont.FreeTypeFont, 'getsize'):
        def getsize(self, text, *a, **k):
            l, t, r, b = self.getbbox(text, *a, **k); return r - l, b - t
        ImageFont.FreeTypeFont.getsize = getsize
    import torch.nn as nn, models.common as C
    patched = [n for n in dir(C) if isinstance(getattr(C, n), type) and 'default_act' in getattr(C, n).__dict__]
    for n in patched: getattr(C, n).default_act = nn.ReLU6()
    assert 'Conv' in patched and 'RepConvN' in patched, patched
    print('ReLU6 on:', ', '.join(patched), flush=True)
    if slim:
        import coral_slim; coral_slim.install(slim)
    return repo


In [ ]:
%%writefile /content/recipe/coral_init.py
#!/usr/bin/env python3
"""Build the starting checkpoint for slim-head training: a fresh YOLOv9-s with the slim head, with every weight whose
name and shape match copied from a trained 17-class ReLU6 checkpoint (backbone, neck, aux head). The slim main-head
towers (cv4/cv5) start from random init; stage A trains them with the backbone frozen.
usage: coral_init.py --repo /abs/yolov9 --slim distlite_cls64 --src yolov9-s-relu6-best.pt --out init_slim.pt [--nc 17]"""
import argparse, copy, os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import coral_patch
ap = argparse.ArgumentParser()
ap.add_argument('--repo', required=True); ap.add_argument('--slim', default='distlite_cls64')
ap.add_argument('--src', required=True); ap.add_argument('--out', required=True); ap.add_argument('--nc', type=int, default=17)
a = ap.parse_args()
src, out = os.path.abspath(a.src), os.path.abspath(a.out)
repo = coral_patch.setup(a.repo, a.slim)
import torch, coral_slim, models.yolo as Y
from utils.general import intersect_dicts
if not hasattr(Y, 'DualDDetectEdgeTPU'): Y.DualDDetectEdgeTPU = coral_slim._BASE   # v1.0 release checkpoints pickle this name
ck = torch.load(src, map_location='cpu'); sm = (ck.get('ema') or ck['model']).float()
torch.manual_seed(0)
m = Y.Model(f'{repo}/models/detect/yolov9-s.yaml', ch=3, nc=a.nc)
h = m.model[-1]; assert type(h).__name__ == 'DualDDetectSlim' and h.slim_arch == a.slim
sd = m.state_dict(); csd = intersect_dicts(sm.state_dict(), sd); m.load_state_dict(csd, strict=False)
fresh = sorted({'.'.join(k.split('.')[:4]) for k in sd if k not in csd})
print(f'copied {len(csd)}/{len(sd)} tensors from {src}; fresh: {fresh}', flush=True)
hp = f'model.{len(m.model) - 1}.'
assert all(k.startswith((hp + 'cv4', hp + 'cv5')) for k in fresh), 'unexpected fresh tensors outside the slim head'
m.eval()
with torch.no_grad(): y = m(torch.zeros(1, 3, 320, 320))
assert tuple(y[0][1].shape) == (1, 4 + a.nc, 2100), y[0][1].shape
m.train()
torch.save({'model': copy.deepcopy(m).half(), 'ema': None, 'updates': None, 'optimizer': None, 'epoch': -1,
            'best_fitness': None, 'slim_arch': a.slim, 'init': src}, out)
print('wrote', out, flush=True)


In [ ]:
%%writefile /content/recipe/coral_train.py
#!/usr/bin/env python3
"""Train YOLOv9-s with ReLU6 and a slim EdgeTPU head. Wraps yolov9/train_dual.py, which stays unmodified.
usage:  coral_train.py --repo /abs/yolov9 --slim distlite_cls64 [train_dual.py args ...]   (use ABSOLUTE paths)
resume: coral_train.py --repo /abs/yolov9 --slim distlite_cls64 --resume /abs/run/weights/last.pt
On top of coral_patch.setup():
  * resume refuses a finished (optimizer-stripped) checkpoint, and keeps mosaic off when resuming inside the
    close-mosaic phase (train_dual only switches it off at exactly epoch == epochs - close_mosaic)
  * when train_dual finishes and strips weights/last.pt, a full copy is kept first as weights/last_resume.pt
    (optimizer + EMA + epoch), so the final state can still be resumed or audited later."""
import argparse, os, runpy, shutil, sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import coral_patch
ap = argparse.ArgumentParser()
ap.add_argument('--repo', required=True)
ap.add_argument('--slim', default='distlite_cls64', help='slim main head: distlite_cls64 (released) | cls64 | "" = stock')
a, rest = ap.parse_known_args()
coral_patch.setup(a.repo, a.slim)
import torch

if '--resume' in rest:
    import yaml
    last = rest[rest.index('--resume') + 1]
    if not os.path.isabs(last): raise SystemExit('--resume needs an absolute path to last.pt')
    ck = torch.load(last, map_location='cpu')
    if ck.get('epoch', -1) < 0 or ck.get('optimizer') is None:
        raise SystemExit(f'{last} is a finished (stripped) checkpoint: nothing to resume. Do not restart it with --weights.')
    o = yaml.safe_load(open(os.path.join(os.path.dirname(os.path.dirname(last)), 'opt.yaml')))
    start = ck['epoch'] + 1; del ck
    print(f"resuming at epoch {start}/{o['epochs']} (close_mosaic {o['close_mosaic']})", flush=True)
    if o['close_mosaic'] and start > o['epochs'] - o['close_mosaic']:
        import utils.dataloaders as D
        orig, state = D.create_dataloader, {'first': True}
        def cdl(*x, **kw):
            loader, ds = orig(*x, **kw)
            if state['first']: ds.mosaic = False; state['first'] = False; print('resume: mosaic off (close-mosaic phase)', flush=True)
            return loader, ds
        D.create_dataloader = cdl

import utils.general as G   # train_dual's `from utils.general import strip_optimizer` picks up this wrapper
_strip = G.strip_optimizer
def _strip_keep(f='best.pt', s=''):
    if os.path.basename(str(f)) == 'last.pt':
        k = os.path.join(os.path.dirname(str(f)), 'last_resume.pt')
        shutil.copy2(f, k + '.tmp'); os.replace(k + '.tmp', k)
        print(f'kept {k} (optimizer + EMA + epoch) before stripping {f}', flush=True)
    return _strip(f, s)
G.strip_optimizer = _strip_keep
sys.argv = ['train_dual.py'] + rest
runpy.run_path('train_dual.py', run_name='__main__')


In [ ]:
%%writefile /content/recipe/coral_val.py
#!/usr/bin/env python3
"""Run yolov9/val_dual.py on a slim-head checkpoint (coral_slim must be importable to unpickle it).
usage: coral_val.py --repo /abs/yolov9 [val_dual.py args ...]"""
import argparse, os, runpy, sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import coral_patch
ap = argparse.ArgumentParser(); ap.add_argument('--repo', required=True); a, rest = ap.parse_known_args()
coral_patch.setup(a.repo)
import coral_slim  # noqa: F401  (class lookup for unpickling)
sys.argv = ['val_dual.py'] + rest
runpy.run_path('val_dual.py', run_name='__main__')


In [ ]:
%%writefile /content/recipe/hyp.slim-s1.yaml
# stage A (slim-head warm-up, backbone frozen): low LR, no mosaic, short warm-up without a bias-LR kick
# "scratch-low"-style fine-tuning hyps for YOLOv9-s @320
lr0: 0.005
lrf: 0.2
momentum: 0.937
weight_decay: 0.0005
warmup_epochs: 0.3
warmup_momentum: 0.8
warmup_bias_lr: 0.0
box: 7.5
cls: 0.5
cls_pw: 1.0
obj: 0.7
obj_pw: 1.0
dfl: 1.5
iou_t: 0.20
anchor_t: 5.0
fl_gamma: 0.0
hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 0.0
translate: 0.1
scale: 0.5        # was 0.9 (scratch-high); small model at 320
shear: 0.0
perspective: 0.0
flipud: 0.0
fliplr: 0.5
mosaic: 0.0
mixup: 0.0        # was 0.15
copy_paste: 0.0   # was 0.3 (needs segments; our labels are boxes only)


In [ ]:
%%writefile /content/recipe/hyp.slim-s2.yaml
# stage B (main run, all layers): fine-tuning LR, no mosaic, short warm-up without a bias-LR kick
# "scratch-low"-style fine-tuning hyps for YOLOv9-s @320
lr0: 0.002
lrf: 0.05
momentum: 0.937
weight_decay: 0.0005
warmup_epochs: 0.3
warmup_momentum: 0.8
warmup_bias_lr: 0.0
box: 7.5
cls: 0.5
cls_pw: 1.0
obj: 0.7
obj_pw: 1.0
dfl: 1.5
iou_t: 0.20
anchor_t: 5.0
fl_gamma: 0.0
hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 0.0
translate: 0.1
scale: 0.5        # was 0.9 (scratch-high); small model at 320
shear: 0.0
perspective: 0.0
flipud: 0.0
fliplr: 0.5
mosaic: 0.0
mixup: 0.0        # was 0.15
copy_paste: 0.0   # was 0.3 (needs segments; our labels are boxes only)


## Step 5: Data set

Set `DATA_YAML` to use your own YOLO-format data (17 classes in the `LABELS` order). Otherwise this cell downloads COCO 2017 (about 20 GB, 15–30 minutes) and keeps only the 17 classes. Images with none of them stay in as background images.


In [ ]:
if not DATA_YAML:
    C = Path(COCO_DIR)
    if not (C / 'coco17.yaml').exists():
        C.parent.mkdir(parents=True, exist_ok=True)
        if not (C / 'labels').exists():
            subprocess.run(f'cd {C.parent} && curl -fsSL -o coco2017labels.zip https://github.com/ultralytics/yolov5/releases/download/v1.0/coco2017labels.zip '
                           f'&& unzip -q -o coco2017labels.zip && rm coco2017labels.zip', shell=True, check=True)
        for split in ('val2017', 'train2017'):
            if not (C / 'images' / split).exists():
                subprocess.run(f'mkdir -p {C}/images && cd {C}/images && curl -fsSL -o {split}.zip http://images.cocodataset.org/zips/{split}.zip '
                               f'&& unzip -q {split}.zip && rm {split}.zip', shell=True, check=True)
        remap = {old: new for new, old in enumerate(COCO80_IDS)}
        for split in ('train2017', 'val2017'):
            n_in = n_out = 0
            for f in (C / 'labels' / split).glob('*.txt'):
                rows = [l.split() for l in f.read_text().splitlines() if l.strip()]
                keep = [' '.join([str(remap[int(r[0])])] + r[1:5]) for r in rows if int(r[0]) in remap]
                n_in += len(rows); n_out += len(keep)
                f.write_text('\n'.join(keep) + ('\n' if keep else ''))
            print(f'{split}: kept {n_out} of {n_in} boxes')
        for p in C.glob('labels/*.cache'): p.unlink()
        names = '\n'.join(f'  {i}: {n}' for i, n in enumerate(LABELS))
        (C / 'coco17.yaml').write_text(f'path: {C}\ntrain: train2017.txt\nval: val2017.txt\nnc: {len(LABELS)}\nnames:\n{names}\n')
    DATA_YAML = str(C / 'coco17.yaml')
print('data:', DATA_YAML); print(open(DATA_YAML).read())


## Step 6: Starting checkpoint


In [ ]:
INIT = f'{RUN_ROOT}/init_{SLIM}.pt'
if not os.path.exists(INIT):
    src = INIT_SRC
    if src.startswith('http'):
        src = f'{WORK}/{Path(INIT_SRC).name}'
        if not os.path.exists(src): subprocess.run(['curl', '-fL', '--retry', '5', '-o', src, INIT_SRC], check=True)
    subprocess.run([PY, f'{RECIPE}/coral_init.py', '--repo', REPO, '--slim', SLIM, '--src', src, '--out', INIT,
                    '--nc', str(len(LABELS))], check=True)
print('init:', INIT)


## Step 7: Training helper

`run_stage()` decides between a fresh start, a true resume, and "already finished", from the checkpoints in the run folder. Re-running a stage cell after a disconnect continues where it stopped.


In [ ]:
import torch, yaml
for p in (REPO, RECIPE):   # checkpoints are pickles of yolov9 + coral_slim classes
    if p not in sys.path: sys.path.insert(0, p)

def ck_info(p):
    ck = torch.load(p, map_location='cpu', weights_only=False)   # our own files only (full pickle)
    return ck.get('epoch', -1), ck.get('optimizer') is not None

def run_stage(name, epochs, init, hyp, extra=()):
    run = f'{RUN_ROOT}/{name}'; w = f'{run}/weights'
    base = [PY, f'{RECIPE}/coral_train.py', '--repo', REPO, '--slim', SLIM]
    if os.path.exists(f'{run}/opt.yaml') and os.path.exists(f'{w}/last.pt'):
        rec = yaml.safe_load(open(f'{run}/opt.yaml'))['epochs']
        if rec != epochs:   # schedule freeze
            raise SystemExit(f'{name} was started with epochs={rec}, not {epochs}. Changing the total mid-run stretches '
                             f'the LR schedule: set the epochs back to {rec}, or start a new run name.')
        e, has_opt = ck_info(f'{w}/last.pt')
        if e < 0 or not has_opt:
            fin = ck_info(f'{w}/last_resume.pt')[0] if os.path.exists(f'{w}/last_resume.pt') else None
            print(f'{name}: finished (last.pt is stripped; final epoch {fin}). Nothing to do.'); return run
        print(f'{name}: true resume at epoch {e + 1}/{epochs}')
        cmd = base + ['--resume', f'{w}/last.pt']
    else:
        print(f'{name}: fresh start from {init}')
        cmd = base + ['--weights', init, '--cfg', 'models/detect/yolov9-s.yaml', '--data', DATA_YAML, '--hyp', hyp,
                      '--epochs', str(epochs), '--close-mosaic', str(epochs), '--img', str(IMG), '--batch-size', str(BATCH),
                      '--workers', str(WORKERS), '--device', DEVICE, '--optimizer', 'SGD', '--min-items', '0',
                      '--project', RUN_ROOT, '--name', name, '--exist-ok', *extra]
    print('$', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    return run


## Step 8: Stage A, slim-head warm-up (backbone frozen)


In [ ]:
RUN_A = run_stage(f'{RUN_NAME}_A', STAGE_A_EPOCHS, INIT, f'{RECIPE}/hyp.slim-s1.yaml', extra=['--freeze', '9'])


## Step 9: Stage B, main run

Starts once from stage A's `best.pt`; every later bout resumes the same run. `EPOCHS_TOTAL` is frozen from here on. On a T4 with the full COCO-17 train set, expect roughly 20–25 minutes per epoch at batch 64.


In [ ]:
RUN_B = run_stage(f'{RUN_NAME}_B', EPOCHS_TOTAL, f'{RUN_A}/weights/best.pt', f'{RECIPE}/hyp.slim-s2.yaml')


## Step 10: Validate


In [ ]:
BEST = f'{RUN_B}/weights/best.pt'
subprocess.run([PY, f'{RECIPE}/coral_val.py', '--repo', REPO, '--weights', BEST, '--data', DATA_YAML, '--img', str(IMG),
                '--batch-size', '32', '--conf-thres', '0.001', '--iou-thres', '0.7', '--device', DEVICE,
                '--project', RUN_ROOT, '--name', f'{RUN_NAME}_val', '--exist-ok'], check=True)


## Step 11: Package

`best.pt` goes to the convert notebook. `last_resume.pt` (or a resumable `last.pt`) holds the optimizer, EMA and epoch for anyone who wants to resume or audit the run; keep it with the run folder.


In [ ]:
PKG = Path(RUN_ROOT) / f'{RUN_NAME}_package'; PKG.mkdir(exist_ok=True)
w = Path(RUN_B) / 'weights'
shutil.copy2(w / 'best.pt', PKG / 'best.pt')
for n in ('last_resume.pt', 'last.pt'):
    if (w / n).exists():
        e, has_opt = ck_info(w / n)
        if e >= 0 and has_opt:
            shutil.copy2(w / n, PKG / 'last_resume.pt'); print(f'resumable state: {n} (epoch {e})'); break
for n in ('results.csv', 'opt.yaml', 'hyp.yaml'):
    if (Path(RUN_B) / n).exists(): shutil.copy2(Path(RUN_B) / n, PKG / n)
shutil.copy2(f'{RECIPE}/coral_slim.py', PKG / 'coral_slim.py')
print('package:', PKG, sorted(p.name for p in PKG.iterdir()))
